In [1]:
import pandas as pd
from src.ab_readout import ab_readout

df = pd.read_csv("data/clean/ab_clean.csv")
users_per_day = len(df) / df["date"].nunique()

result = ab_readout(df, group_col="group", metric_col="converted",
                    control="control", treatment="treatment",
                    value_per_conversion=50, annual_users=users_per_day * 365)

--------------------------------------------------------------
A/B TEST READOUT
--------------------------------------------------------------
1. Sample ratio   control 145,274 | treatment 145,310 | SRM p = 0.9468 OK
2. Conversion     control 12.039% | treatment 11.881%
   Difference     -0.158 pp (-1.31% relative)
3. Frequentist    95% CI -0.394 to +0.078 pp | z = -1.31 | p = 0.1899
4. Power          MDE 0.340 pp (2.83% relative) at 80% power
5. Bayesian       P(treatment better) = 9.5%
                  expected loss per 10k users: ship 16.3 | keep 0.5
6. £ per year     expected -363,131 | range -906,900 to +184,099
                  regret: ship £375,658 | keep £12,526
--------------------------------------------------------------
DECISION: KEEP CONTROL - no significant difference, and shipping carries more risk (16.3 vs 0.5 conversions lost per 10k users). This test could detect lifts of 2.8% or more.
--------------------------------------------------------------


In [3]:
def make_test(n_c, x_c, n_t, x_t):
    return pd.DataFrame({
        "group": ["control"] * n_c + ["treatment"] * n_t,
        "converted": [1] * x_c + [0] * (n_c - x_c) + [1] * x_t + [0] * (n_t - x_t)})

print("SCENARIO A: a genuine winner")
ab_readout(make_test(50000, 6000, 50000, 6400), "group", "converted", "control", "treatment")

print("\nSCENARIO B: broken 50/50 split")
_ = ab_readout(make_test(50000, 6000, 52000, 6240), "group", "converted", "control", "treatment")

SCENARIO A: a genuine winner
--------------------------------------------------------------
A/B TEST READOUT
--------------------------------------------------------------
1. Sample ratio   control 50,000 | treatment 50,000 | SRM p = 1.0000 OK
2. Conversion     control 12.000% | treatment 12.800%
   Difference     +0.800 pp (+6.67% relative)
3. Frequentist    95% CI +0.391 to +1.209 pp | z = 3.84 | p = 0.0001
4. Power          MDE 0.582 pp (4.85% relative) at 80% power
5. Bayesian       P(treatment better) = 100.0%
                  expected loss per 10k users: ship 0.0 | keep 80.1
--------------------------------------------------------------
DECISION: SHIP - treatment is better and the whole confidence interval is above zero.
--------------------------------------------------------------

SCENARIO B: broken 50/50 split
--------------------------------------------------------------
A/B TEST READOUT
--------------------------------------------------------------
1. Sample ratio   contro

In [4]:
pd.DataFrame([{k: v for k, v in result.items() if k != "money"}]).to_csv("outputs/readout_summary.csv", index=False)